# Module 08 — GPU Hardware Selection

This notebook compares representative T4, A100, H100, H200, and B200 constraints, estimates model fit, calculates memory-streaming lower bounds, and optionally inspects the attached Lightning GPU.

The central rule is: **capacity is a gate; bandwidth, compute, interconnect, and cost determine speed after the workload fits.**

## Learning goals

- compare GPU capacity, memory bandwidth, precision, and interconnect constraints;
- estimate whether weights, KV cache, and runtime headroom fit;
- compare idealized memory-streaming lower bounds;
- inspect the actual GPU and topology available to the notebook.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd() / 'code' / 'module-08', Path.cwd()]
module_dir = next(path for path in candidates if (path / 'hardware_selection.py').exists())
sys.path.insert(0, str(module_dir))

from hardware_selection import (
    GPU_SPECS, estimate_budget, print_budget, print_lower_bounds,
    print_specs, probe_device,
)

print('Companion module:', module_dir / 'hardware_selection.py')

## 1. Compare representative GPU constraints

These are published-style reference values for specific variants, not application benchmarks. Check the exact SKU before deployment.

In [ ]:
print_specs()

In [ ]:
import matplotlib.pyplot as plt

names = [spec.name for spec in GPU_SPECS]
memory = [spec.memory_gb for spec in GPU_SPECS]
bandwidth = [spec.bandwidth_gbs / 1000 for spec in GPU_SPECS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(names, memory, color='#38bdf8')
axes[0].set_title('Local GPU memory')
axes[0].set_ylabel('GB')
axes[0].tick_params(axis='x', rotation=35)
axes[0].grid(axis='y', alpha=0.25)
axes[1].bar(names, bandwidth, color='#a78bfa')
axes[1].set_title('Peak memory bandwidth')
axes[1].set_ylabel('TB/s')
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## 2. Estimate model and KV-cache fit

This is transparent planning arithmetic. Real checkpoints, quantization metadata, allocator behavior, workspaces, and vLLM settings must be measured.

In [ ]:
parameters_b = 7
weight_bytes = 2.0       # FP16/BF16
reserve_gb = 3.0
kv_gb_per_1k_tokens = 0.125
tokens = 4_096

print_budget(parameters_b, weight_bytes, reserve_gb, kv_gb_per_1k_tokens, tokens)

In [ ]:
for label, bytes_per_weight in [('FP16/BF16', 2.0), ('INT8', 1.0), ('INT4', 0.5)]:
    budget = estimate_budget(
        parameters_b, bytes_per_weight, reserve_gb,
        kv_gb_per_1k_tokens, tokens,
    )
    fits = [spec.name for spec in GPU_SPECS if budget['total_gb'] <= spec.usable_gb]
    print(f'{label:10s}: {budget["total_gb"]:6.2f} GB; fits: {fits}')

## 3. Idealized memory-streaming lower bounds

A 14 GB read is a teaching example. It is not a token-generation latency prediction.

In [ ]:
print_lower_bounds(14.0)

## 4. Inspect the actual Lightning GPU

Run the next cell on a CUDA runtime. It reports all visible GPUs, free VRAM, compute capability, and SM count.

In [ ]:
try:
    probe_device(show_topology=False)
except RuntimeError as exc:
    print(exc)

In [ ]:
# Optional multi-GPU topology inspection.
try:
    probe_device(show_topology=True)
except RuntimeError as exc:
    print(exc)

## Interpretation checklist

1. Does the workload fit with runtime headroom?
2. Is it likely bandwidth-sensitive, compute-sensitive, or communication-sensitive?
3. Does the selected dtype have an efficient kernel on the chosen GPU?
4. Would independent replicas avoid communication that tensor parallelism would introduce?
5. What end-to-end throughput, latency, memory, and cost measurements remain to be collected?

A peak specification is a ceiling. The request mix, serving configuration, and actual kernels determine how close the workload gets to it.